# 05 — Append-Only Filtering (FilteringPress)

This notebook demonstrates the **append-only filtering** compression
strategy: during generation, each new token is scored and either kept
or skipped before entering the cache. The cache only grows, never
shrinks — compression comes from skipping low-scoring tokens.

This mirrors kvpress's `FilteringPress`, which wraps a scoring press
(like `KeyDiffPress`) and applies a keep/skip threshold at each decode
step.

We run three experiments:
1. **Filtering decisions** — simulate decode steps with per-head
   keep/skip decisions using dense tensors
2. **Cross-validation** — verify our decisions match kvpress's
   `FilteringPress` on the same key sequence
3. **Full pipeline on paged cache** — same filtering but with keys
   living in the Flash Attention paged cache

## Imports and Setup

In [ ]:
import torch
import torch.nn.functional as F
from vllm import _custom_ops as ops

torch.manual_seed(42)
torch.set_grad_enabled(False)

assert torch.cuda.is_available(), "CUDA GPU required"
print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
try:
    from kvpress import KeyDiffPress, FilteringPress
    HAS_KVPRESS = True
    print("kvpress available — will validate against it")
except ImportError:
    HAS_KVPRESS = False
    print("kvpress not installed — skipping cross-validation experiments")

## Configuration

In [ ]:
DTYPE = torch.float16
NUM_KV_HEADS = 4
HEAD_SIZE = 128
BLOCK_SIZE = 16
PREFILL_LEN = 64
NUM_DECODE_STEPS = 32
COMPRESSION_RATIO = 0.5
DEVICE = "cuda"

## Primitives

Cache operations from notebook 02, scoring from notebook 03, plus
`filter_new_token` — the per-head keep/skip decision function used
by the append-only strategy.

In [ ]:
def create_kv_caches_flash(num_blocks, block_size, num_kv_heads, head_size,
                           dtype, device="cuda"):
    key_cache = torch.randn(
        num_blocks, block_size, num_kv_heads, head_size,
        dtype=dtype, device=device,
    )
    value_cache = torch.randn(
        num_blocks, block_size, num_kv_heads, head_size,
        dtype=dtype, device=device,
    )
    return key_cache, value_cache


def build_slot_mapping_for_positions(block_table, positions, block_size):
    logical_block_indices = positions // block_size
    physical_block_indices = block_table[logical_block_indices]
    offsets = positions % block_size
    return physical_block_indices * block_size + offsets


def gather_from_paged_cache(key_cache, value_cache, slot_mapping, block_size):
    block_indices = slot_mapping // block_size
    offsets = slot_mapping % block_size
    keys = key_cache[block_indices, offsets]
    values = value_cache[block_indices, offsets]
    return keys, values


print("Cache functions defined")

In [ ]:
def keydiff_score(keys):
    """Score keys using KeyDiff's key-similarity metric.

    keys: [seq_len, num_kv_heads, head_dim]
    Returns: [num_kv_heads, seq_len]. Higher scores = more important.
    """
    keys_by_head = keys.permute(1, 0, 2)
    normalized = F.normalize(keys_by_head, p=2, dim=-1)
    anchor = normalized.mean(dim=1, keepdim=True)
    scores = -F.cosine_similarity(keys_by_head, anchor, dim=-1)
    return scores


def filter_new_token(scores, total_tokens_seen, compression_ratio):
    """Decide per head whether the newest token survives filtering.

    scores: [num_kv_heads, seq_len] — all tokens scored, including the
        new one at position seq_len - 1.
    total_tokens_seen: logical count of all tokens processed so far
        (including skipped ones). Drives the n_kept calculation.
    compression_ratio: target fraction of tokens to filter out.

    Returns: [num_kv_heads] — boolean. True = keep, False = reject.
    """
    n_kept = max(1, int(total_tokens_seen * (1 - compression_ratio)))
    n_kept = min(n_kept, scores.shape[-1])
    threshold = scores.topk(n_kept, dim=-1, sorted=True).values[:, -1]
    return scores[:, -1] >= threshold


print("Scoring and filtering functions defined")

## Experiment 1 — Filtering Decisions

Simulate a sequence of decode steps. At each step:
1. Append the new token's key to the cached keys
2. Score all keys with KeyDiff
3. Apply the filtering threshold to decide per head whether to keep
   the new token

In [ ]:
all_keys = torch.randn(
    PREFILL_LEN + NUM_DECODE_STEPS, NUM_KV_HEADS, HEAD_SIZE,
    dtype=DTYPE, device=DEVICE,
)

print(f"Prefill tokens:     {PREFILL_LEN}")
print(f"Decode steps:       {NUM_DECODE_STEPS}")
print(f"Compression ratio:  {COMPRESSION_RATIO}")
print(f"Total keys:         {all_keys.shape[0]}")

In [ ]:
cached_keys = all_keys[:PREFILL_LEN].clone()
decisions = []

for step in range(NUM_DECODE_STEPS):
    new_key = all_keys[PREFILL_LEN + step]
    total_tokens_seen = PREFILL_LEN + step + 1

    keys_with_new = torch.cat(
        [cached_keys, new_key.unsqueeze(0)], dim=0,
    )

    scores = keydiff_score(keys_with_new)
    keep_per_head = filter_new_token(scores, total_tokens_seen, COMPRESSION_RATIO)

    decisions.append({
        "step": step,
        "total_seen": total_tokens_seen,
        "cache_len_before": cached_keys.shape[0],
        "keep": keep_per_head.clone(),
        "all_keep": keep_per_head.all().item(),
        "any_keep": keep_per_head.any().item(),
    })

    # Uniform (all-or-nothing) caching: if ANY head wants to keep
    # the token, cache it. The real FilteringPress uses per-head
    # ragged lengths via PaddedTensor, but for decision validation,
    # the scoring and threshold logic is what matters.
    if keep_per_head.any():
        cached_keys = keys_with_new

kept_count = sum(1 for d in decisions if d["any_keep"])
skipped_count = NUM_DECODE_STEPS - kept_count
effective_ratio = skipped_count / (PREFILL_LEN + NUM_DECODE_STEPS)

print(f"\nResults:")
print(f"  Kept:    {kept_count}/{NUM_DECODE_STEPS} decode tokens")
print(f"  Skipped: {skipped_count}/{NUM_DECODE_STEPS} decode tokens")
print(f"  Final cache size: {cached_keys.shape[0]} tokens")
print(f"  Effective compression: {effective_ratio:.2%} of total tokens skipped")
print()

for d in decisions:
    heads_kept = d['keep'].sum().item()
    marker = "KEEP" if d['any_keep'] else "SKIP"
    print(
        f"  step {d['step']:2d}  seen={d['total_seen']:3d}  "
        f"cache={d['cache_len_before']:3d}  "
        f"heads={heads_kept}/{NUM_KV_HEADS}  {marker}"
    )

## Experiment 2 — Cross-Validation Against kvpress

Run the same key sequence through kvpress's `FilteringPress` and verify
that the per-head keep/skip decisions match our standalone implementation.

This requires a mock `nn.Module` with a `layer_idx` attribute, and
`position_ids` in kwargs — the only external dependencies of
`FilteringPress.compress()`.

In [ ]:
if not HAS_KVPRESS:
    print("Skipping — kvpress not installed")
else:
    from types import SimpleNamespace

    mock_module = SimpleNamespace(layer_idx=0, head_dim=HEAD_SIZE)

    fp = FilteringPress(
        base_press=KeyDiffPress(),
        target_compression_ratio=COMPRESSION_RATIO,
    )

    kvpress_keys = all_keys[:PREFILL_LEN].permute(1, 0, 2).unsqueeze(0).clone()
    kvpress_values = torch.zeros_like(kvpress_keys)

    mismatches = 0
    fp.reset()

    for step in range(NUM_DECODE_STEPS):
        new_key = all_keys[PREFILL_LEN + step]
        total_tokens_seen = PREFILL_LEN + step + 1

        new_key_kvpress = new_key.unsqueeze(0).unsqueeze(0)
        new_key_kvpress = new_key_kvpress.permute(0, 2, 1, 3)
        keys_in = torch.cat([kvpress_keys, new_key_kvpress], dim=2)
        values_in = torch.cat(
            [kvpress_values, torch.zeros_like(new_key_kvpress)], dim=2,
        )

        position_ids = torch.arange(
            total_tokens_seen, device=DEVICE,
        ).unsqueeze(0)

        keys_out, values_out = fp.compress(
            module=mock_module,
            hidden_states=None,
            keys=keys_in,
            values=values_in,
            attentions=None,
            kwargs={"position_ids": position_ids},
        )

        kvpress_kept = keys_out.shape[2] >= keys_in.shape[2]
        our_kept = decisions[step]["any_keep"]

        if kvpress_kept != our_kept:
            mismatches += 1
            print(
                f"  MISMATCH step {step}: ours={our_kept}, "
                f"kvpress={kvpress_kept}"
            )

        kvpress_keys = keys_out
        kvpress_values = values_out

    if mismatches == 0:
        print(
            f"All {NUM_DECODE_STEPS} filtering decisions match "
            f"kvpress FilteringPress"
        )
    else:
        print(f"\n{mismatches}/{NUM_DECODE_STEPS} decisions differ")

## Experiment 3 — Full Pipeline on Paged Cache

End-to-end: keys live in the Flash Attention paged cache. At each decode
step, gather cached keys, score together with the new token, and decide
whether to write the new token to the cache.

In [ ]:
all_values = torch.randn(
    PREFILL_LEN + NUM_DECODE_STEPS, NUM_KV_HEADS, HEAD_SIZE,
    dtype=DTYPE, device=DEVICE,
)

total_len = PREFILL_LEN + NUM_DECODE_STEPS
num_blocks = (total_len + BLOCK_SIZE - 1) // BLOCK_SIZE + 4
key_cache, value_cache = create_kv_caches_flash(
    num_blocks, BLOCK_SIZE, NUM_KV_HEADS, HEAD_SIZE, DTYPE, DEVICE,
)

num_seq_blocks = (total_len + BLOCK_SIZE - 1) // BLOCK_SIZE
block_table = torch.arange(num_seq_blocks, dtype=torch.long, device=DEVICE)
k_scale = torch.tensor(1.0, dtype=torch.float32, device=DEVICE)
v_scale = torch.tensor(1.0, dtype=torch.float32, device=DEVICE)

prefill_positions = torch.arange(PREFILL_LEN, dtype=torch.long, device=DEVICE)
prefill_slots = build_slot_mapping_for_positions(
    block_table, prefill_positions, BLOCK_SIZE,
)
ops.reshape_and_cache_flash(
    all_keys[:PREFILL_LEN], all_values[:PREFILL_LEN],
    key_cache, value_cache,
    prefill_slots, "auto", k_scale, v_scale,
)

print(f"Prefilled {PREFILL_LEN} tokens into paged cache")

In [ ]:
cache_len = PREFILL_LEN
paged_decisions = []

for step in range(NUM_DECODE_STEPS):
    total_tokens_seen = PREFILL_LEN + step + 1
    new_key = all_keys[PREFILL_LEN + step].unsqueeze(0)
    new_value = all_values[PREFILL_LEN + step].unsqueeze(0)

    cached_positions = torch.arange(cache_len, dtype=torch.long, device=DEVICE)
    cached_slots = build_slot_mapping_for_positions(
        block_table, cached_positions, BLOCK_SIZE,
    )
    cached_keys, _ = gather_from_paged_cache(
        key_cache, value_cache, cached_slots, BLOCK_SIZE,
    )

    keys_with_new = torch.cat([cached_keys, new_key], dim=0)
    scores = keydiff_score(keys_with_new)

    keep_per_head = filter_new_token(scores, total_tokens_seen, COMPRESSION_RATIO)

    paged_decisions.append(keep_per_head.any().item())

    if keep_per_head.any():
        new_position = torch.tensor([cache_len], dtype=torch.long, device=DEVICE)
        new_slot = build_slot_mapping_for_positions(
            block_table, new_position, BLOCK_SIZE,
        )
        ops.reshape_and_cache_flash(
            new_key, new_value,
            key_cache, value_cache,
            new_slot, "auto", k_scale, v_scale,
        )
        cache_len += 1

dense_decisions = [d["any_keep"] for d in decisions]

mismatches = sum(
    1 for p, d in zip(paged_decisions, dense_decisions) if p != d
)

print(f"Final cache size: {cache_len} tokens")
print(f"Kept: {sum(paged_decisions)}/{NUM_DECODE_STEPS} decode tokens")
print()

if mismatches == 0:
    print(
        f"All {NUM_DECODE_STEPS} decisions match between "
        f"paged-cache and dense-tensor pipelines"
    )
else:
    print(f"{mismatches}/{NUM_DECODE_STEPS} decisions differ")
    for i, (p, d) in enumerate(zip(paged_decisions, dense_decisions)):
        if p != d:
            print(f"  step {i}: paged={p}, dense={d}")

## Notes and Next Steps

**Filtering decisions validated.** The standalone `filter_new_token`
function makes the same per-head keep/skip decisions as kvpress's
`FilteringPress`, and these decisions are identical whether scoring
from dense tensors or from gathered paged cache data.

**What this enables:** With scoring and filtering validated on the
Flash Attention cache layout, the next step is integrating this into
vLLM's `Attention.forward()`. The integration will:
1. Gather cached keys after `do_kv_cache_update`
2. Score with `keydiff_score`
3. Filter the slot mapping with `filter_new_token`
4. Track logical positions for RoPE separately from cache positions